# 01 — Quickstart with Brightway

**Audience:** Premise users who already have a licensed ecoinvent database in a Brightway project.

**Prerequisites:** Python 3.10+, `premise[bw25]` or `premise[bw2]`, an ecoinvent 3.12 cutoff database and biosphere, and `PREMISE_KEY` set in the environment.

**Learning goals:** configure `NewDatabase`, apply all or selected sector updates, and write a scenario database to Brightway.


## Outline

1. Configure and validate the Brightway project.
2. Define one IAM scenario.
3. Build and update the database.
4. Export it with a deterministic name.


## 1. Configure the project

Keep project-specific values together. Premise never needs the IAM key to be written into the notebook itself.


In [ ]:
import os

import bw2data as bd

from premise import NewDatabase

PROJECT = "ecoinvent-3.12-cutoff"
SOURCE_DATABASE = "ecoinvent-3.12-cutoff"
BIOSPHERE_DATABASE = "ecoinvent-3.12-biosphere"
SOURCE_VERSION = "3.12"
PREMISE_KEY = os.environ.get("PREMISE_KEY")

if not PREMISE_KEY:
    raise RuntimeError("Set PREMISE_KEY before running this tutorial.")

bd.projects.set_current(PROJECT)
missing = [
    name for name in (SOURCE_DATABASE, BIOSPHERE_DATABASE) if name not in bd.databases
]
if missing:
    raise ValueError(f"Missing Brightway databases: {missing}")


## 2. Define the scenario

A scenario is one IAM model, pathway, and year. Add more dictionaries to build several databases in one run.


In [ ]:
SCENARIOS = [
    {"model": "remind", "pathway": "SSP2-NDC", "year": 2030},
]


## 3. Build and update

`ndb.update()` applies every available sector transformation. For a faster targeted run, pass a list such as `["electricity", "steel"]` instead.


In [ ]:
ndb = NewDatabase(
    scenarios=[scenario.copy() for scenario in SCENARIOS],
    source_db=SOURCE_DATABASE,
    source_version=SOURCE_VERSION,
    source_type="brightway",
    system_model="cutoff",
    biosphere_name=BIOSPHERE_DATABASE,
    key=PREMISE_KEY,
)

SECTORS = None  # Example: ["electricity", "steel"]
if SECTORS is None:
    ndb.update()
else:
    ndb.update(SECTORS)


## 4. Export to Brightway

One output name is required for each scenario. Explicit names make later analyses reproducible.


In [ ]:
OUTPUT_NAMES = ["premise-remind-ssp2-ndc-2030"]
ndb.write_db_to_brightway(name=OUTPUT_NAMES)

assert all(name in bd.databases for name in OUTPUT_NAMES)
OUTPUT_NAMES


## Pitfalls and extension

- `SOURCE_DATABASE` must exactly match a database registered in the active project.
- `BIOSPHERE_DATABASE` must match the biosphere used by that ecoinvent installation.
- A full update is expensive; use selected sectors while learning or debugging.
- Extension: add 2040 and 2050 to `SCENARIOS`, then provide three unique output names.

## Exercise

Prepare a 2040 scenario that updates only electricity and steel. Predict the number of output databases before running it.


In [ ]:
exercise_scenarios = [
    {"model": "remind", "pathway": "SSP2-NDC", "year": 2040},
]
exercise_sectors = ["electricity", "steel"]
exercise_output_names = ["premise-remind-ssp2-ndc-2040"]
